In [62]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['brian-team/brian2genn']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,006d6af0835e5c18c30850d29d45ec59da41216a,brian-team_brian2genn


In [63]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  1%|▊                                                                                  | 1/97 [00:01<01:39,  1.03s/it]

Got Errors {'038b361e568b6aafa5e2dc258e7f71b7e0b6e176': 'Key 038b361e568b6aafa5e2dc258e7f71b7e0b6e176 not found in /da5_fast/All.sha1c/commit_3.tch'}


  2%|█▋                                                                                 | 2/97 [00:02<01:38,  1.04s/it]

Got Errors {'048cb7d87285cdceae5c6e4c5aae35afdd49bb9a': 'Key 048cb7d87285cdceae5c6e4c5aae35afdd49bb9a not found in /da5_fast/All.sha1c/commit_4.tch'}


 10%|████████▍                                                                         | 10/97 [00:10<01:30,  1.04s/it]

...

Got Errors {'e1b2906956f872b4c44545da86d0edf5c4672fe1': 'Key e1b2906956f872b4c44545da86d0edf5c4672fe1 not found in /da5_fast/All.sha1c/commit_97.tch'}


 98%|████████████████████████████████████████████████████████████████████████████████▎ | 95/97 [01:39<00:02,  1.04s/it]

Got Errors {'fa983897c4ce81a8209cb9b09b796a2c9e8f01f4': 'Key fa983897c4ce81a8209cb9b09b796a2c9e8f01f4 not found in /da5_fast/All.sha1c/commit_122.tch'}


100%|██████████████████████████████████████████████████████████████████████████████████| 97/97 [01:41<00:00,  1.05s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,006d6af0835e5c18c30850d29d45ec59da41216a,50eea9ae261c38328953b23ff9441cb087c86a42,[11d0a7a0915b941aa8c6d0003c51f468046d2ace],Thomas Nowotny <t.nowotny@sussex.ac.uk>,1440518697,+0100,Thomas Nowotny <t.nowotny@sussex.ac.uk>,1440518697,+0100,Fixed a bug where the not_refractory variable ...,006d6af0835e5c18c30850d29d45ec59da41216a,brian-team_brian2genn
1,017d4fa7173396ea48340b2702d49e21ee43ec02,b0f135ffa35123d3b37dab930eb5d51cd4025b77,[eeeced22eeae75eba73f584e66bf1c02e8924e1b],Marcel Stimberg <marcel.stimberg@inserm.fr>,1495641860,+0200,Marcel Stimberg <marcel.stimberg@inserm.fr>,1495641860,+0200,Create a conda package on travis\n,017d4fa7173396ea48340b2702d49e21ee43ec02,brian-team_brian2genn


In [64]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '006d6af0835e5c18c30850d29d45ec59da41216a', 'tree': '50eea9ae261c38328953b23ff9441cb087c86a42', 'parent': ['11d0a7a0915b941aa8c6d0003c51f468046d2ace'], 'author': 'Thomas Nowotny <t.nowotny@sussex.ac.uk>', 'author_time': 1440518697, 'author_tz': '+0100', 'committer': 'Thomas Nowotny <t.nowotny@sussex.ac.uk>', 'committer_time': 1440518697, 'committer_tz': '+0100', 'message': 'Fixed a bug where the not_refractory variable was not set after a spike was detected. This closes #21 as it was the only remaining issue. Supporting subgroups will be a searate question.\n'}


In [65]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [66]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,brian-team_brian2genn,006d6af0835e5c18c30850d29d45ec59da41216a,Thomas Nowotny <t.nowotny@sussex.ac.uk>,1440518697,Fixed a bug where the not_refractory variable ...


In [67]:
#mode a means append, so you have all your projects in the same file
yournetid='kbc177'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github